# 12. Local validation of the promotion-policy boundary

This notebook performs one focused final check of the main empirical mechanism.

It:

1. loads the final demand-only artifact created by Notebook 10;
2. rebuilds the dynamic schedule system on a fine contract-generosity grid;
3. validates the interior boundary around \(\alpha\approx 2.55\) at weekly capacity \(B=2\);
4. verifies consistency with the coarser grid at overlapping values;
5. identifies whether each local jump is caused by a dynamic-policy switch, a myopic-policy switch, or both;
6. saves a compact results table, transition table, figures, and decomposition artifact.

Run Notebook 10 before running this notebook.

## 1. Imports, paths, and frozen validation design

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Mapping, Sequence
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        root
        for root in [CURRENT_DIR, CURRENT_DIR.parent]
        if (root / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. Run this notebook from the "
        "repository root or the notebooks directory."
    )

for import_directory in [
    PROJECT_ROOT / "src",
    PROJECT_ROOT,
    CURRENT_DIR,
]:
    if (
        import_directory.is_dir()
        and str(import_directory) not in sys.path
    ):
        sys.path.insert(0, str(import_directory))

import corrected_promotion_analysis as cpa

from corrected_promotion_analysis import (
    build_schedule_system,
    load_pickle,
    save_pickle,
    schedule_input_fingerprint,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
FIGURE_DIR = PROJECT_ROOT / "results" / "figures"

for directory in [PROCESSED_DIR, TABLE_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Analysis module:", Path(cpa.__file__).resolve())

In [ ]:
# Focused validation design.
VALIDATION_CAPACITY = 2
ALPHA_MIN = 2.40
ALPHA_MAX = 2.65
ALPHA_STEP = 0.01

FINE_ALPHA_GRID = np.round(
    np.arange(
        ALPHA_MIN,
        ALPHA_MAX + 0.5 * ALPHA_STEP,
        ALPHA_STEP,
    ),
    4,
)

SCHEDULE_BATCH_SIZE = 256
MILP_TIME_LIMIT_SECONDS = None
COMPUTE_SECOND_BEST = True

print("Capacity:", VALIDATION_CAPACITY)
print(
    "Fine alpha grid:",
    FINE_ALPHA_GRID[0],
    "to",
    FINE_ALPHA_GRID[-1],
    f"({len(FINE_ALPHA_GRID)} values)",
)

## 2. Load and validate the final Notebook 10 artifact

In [ ]:
ARTIFACT_PATH = (
    PROCESSED_DIR
    / "corrected_policy_optimization_artifact.pkl"
)

if not ARTIFACT_PATH.is_file():
    raise FileNotFoundError(
        f"Missing Notebook 10 artifact: {ARTIFACT_PATH}\n"
        "Run the final minimal Notebook 10 first."
    )

artifact = load_pickle(ARTIFACT_PATH)

required_keys = {
    "selected_washout",
    "economic_profile_mode",
    "draws_by_product",
    "weekly_profiles",
    "action_sets",
    "support_table",
    "schedule_system",
    "policy_results",
    "products",
}

missing_keys = required_keys.difference(artifact)
if missing_keys:
    raise KeyError(
        "Notebook 10 artifact is missing keys: "
        f"{sorted(missing_keys)}"
    )

SELECTED_WASHOUT = int(
    artifact["selected_washout"]
)
ECONOMIC_PROFILE_MODE = str(
    artifact["economic_profile_mode"]
)

draws_by_product = artifact[
    "draws_by_product"
]
weekly_profiles = artifact[
    "weekly_profiles"
]
action_sets = artifact[
    "action_sets"
]
support_table = artifact[
    "support_table"
]
products = list(
    artifact["products"]
)
coarse_policy_results = (
    artifact["policy_results"]
    .copy()
)

planning = artifact[
    "schedule_system"
][
    "planning"
]

if "demand" not in ECONOMIC_PROFILE_MODE.lower():
    raise AssertionError(
        "The loaded artifact does not appear to use the final "
        "weekly-demand specification."
    )

if not all(
    np.allclose(
        np.asarray(
            values["price_factor"],
            dtype=float,
        ),
        1.0,
    )
    for values in weekly_profiles.values()
):
    raise AssertionError(
        "Price factors are not fixed to one."
    )

if not all(
    np.allclose(
        np.asarray(
            values["cost_factor"],
            dtype=float,
        ),
        1.0,
    )
    for values in weekly_profiles.values()
):
    raise AssertionError(
        "Cost factors are not fixed to one."
    )

print("Economic profile:", ECONOMIC_PROFILE_MODE)
print("Selected washout:", SELECTED_WASHOUT)
print("Decision horizon:", planning.decision_horizon)
print("Evaluation horizon:", planning.evaluation_horizon)
print("Products:", len(products))

## 3. Rebuild the dynamic schedule system on the fine grid

The schedule candidates must be rebuilt because candidate pruning depends on the evaluated \(\alpha\)-grid.

In [ ]:
expected_fingerprint = (
    schedule_input_fingerprint(
        draws_by_product=draws_by_product,
        weekly_profiles=weekly_profiles,
        action_sets=action_sets,
        planning=planning,
        alpha_grid=FINE_ALPHA_GRID,
    )
)

cache_path = (
    PROCESSED_DIR
    / (
        "boundary_validation_schedule_system_"
        f"b{VALIDATION_CAPACITY}_"
        f"a{ALPHA_MIN:.2f}_{ALPHA_MAX:.2f}_"
        f"s{ALPHA_STEP:.2f}_"
        f"w{SELECTED_WASHOUT}.pkl"
    )
)

fine_schedule_system = None

if cache_path.is_file():
    cached = load_pickle(cache_path)

    if (
        cached.get("input_fingerprint")
        == expected_fingerprint
    ):
        fine_schedule_system = cached
        print("Loaded current cache:", cache_path.name)

if fine_schedule_system is None:
    fine_schedule_system = (
        build_schedule_system(
            draws_by_product=draws_by_product,
            weekly_profiles=weekly_profiles,
            action_sets=action_sets,
            planning=planning,
            alpha_grid=FINE_ALPHA_GRID,
            batch_size=SCHEDULE_BATCH_SIZE,
        )
    )

    save_pickle(
        fine_schedule_system,
        cache_path,
    )

    print("Built schedule system:", cache_path.name)

## 4. Solve dynamic and myopic policies at every fine-grid value

In [ ]:
summary_rows = []
product_frames = []
weekly_frames = []
fine_schedules = {}

for alpha in FINE_ALPHA_GRID:
    alpha = float(alpha)

    dynamic_solution = (
        cpa.solve_dynamic_category(
            schedule_system=fine_schedule_system,
            alpha=alpha,
            capacity=VALIDATION_CAPACITY,
            compute_second_best=COMPUTE_SECOND_BEST,
            time_limit_seconds=MILP_TIME_LIMIT_SECONDS,
        )
    )

    myopic_solution = (
        cpa.simulate_myopic_category(
            draws_by_product=draws_by_product,
            weekly_profiles=weekly_profiles,
            action_sets=action_sets,
            planning=planning,
            alpha=alpha,
            capacity=VALIDATION_CAPACITY,
        )
    )

    (
        summary,
        product_decomposition,
        weekly_decomposition,
    ) = cpa.analyze_policy_pair(
        dynamic_solution=dynamic_solution,
        myopic_solution=myopic_solution,
        draws_by_product=draws_by_product,
        weekly_profiles=weekly_profiles,
        planning=planning,
        support_table=support_table,
        alpha=alpha,
        capacity=VALIDATION_CAPACITY,
    )

    summary["solver_message"] = (
        dynamic_solution["solver_message"]
    )
    summary[
        "economic_profile_mode"
    ] = ECONOMIC_PROFILE_MODE

    summary_rows.append(summary)
    product_frames.append(
        product_decomposition
    )
    weekly_frames.append(
        weekly_decomposition
    )

    fine_schedules[
        (
            round(alpha, 8),
            VALIDATION_CAPACITY,
        )
    ] = {
        "dynamic": dynamic_solution[
            "schedule_map"
        ],
        "myopic": myopic_solution[
            "schedule_map"
        ],
    }

fine_results = (
    pd.DataFrame(summary_rows)
    .sort_values("alpha")
    .reset_index(drop=True)
)

fine_product_decomposition = (
    pd.concat(
        product_frames,
        ignore_index=True,
    )
)

fine_weekly_decomposition = (
    pd.concat(
        weekly_frames,
        ignore_index=True,
    )
)

if (
    fine_results["vdo"] < -1e-6
).any():
    raise AssertionError(
        "Dynamic profit falls below myopic profit "
        "at one or more fine-grid values."
    )

maximum_vdo_error = float(
    (
        fine_results["vdo"]
        - (
            fine_results["dynamic_profit"]
            - fine_results["myopic_profit"]
        )
    ).abs().max()
)

if maximum_vdo_error > 1e-6:
    raise AssertionError(
        "VDO arithmetic check failed. "
        f"Maximum error: {maximum_vdo_error}"
    )

print("Fine-grid validation passed.")
print("Solver messages:")
for message in fine_results[
    "solver_message"
].drop_duplicates():
    print(" -", message)

display(
    fine_results[
        [
            "alpha",
            "dynamic_profit",
            "myopic_profit",
            "vdo",
            "vdo_percent",
            "dynamic_promotion_count",
            "myopic_promotion_count",
            "action_disagreements",
            "best_second_gap",
        ]
    ]
)

## 5. Verify exact agreement with Notebook 10 at overlapping grid points

In [ ]:
coarse_capacity_results = (
    coarse_policy_results.loc[
        coarse_policy_results[
            "capacity"
        ].eq(
            VALIDATION_CAPACITY
        )
    ]
    .copy()
)

overlap_alphas = sorted(
    set(
        np.round(
            fine_results["alpha"],
            8,
        )
    )
    .intersection(
        set(
            np.round(
                coarse_capacity_results[
                    "alpha"
                ],
                8,
            )
        )
    )
)

overlap_check = (
    fine_results.loc[
        fine_results[
            "alpha"
        ].round(8).isin(
            overlap_alphas
        ),
        [
            "alpha",
            "dynamic_profit",
            "myopic_profit",
            "vdo",
        ],
    ]
    .merge(
        coarse_capacity_results.loc[
            coarse_capacity_results[
                "alpha"
            ].round(8).isin(
                overlap_alphas
            ),
            [
                "alpha",
                "dynamic_profit",
                "myopic_profit",
                "vdo",
            ],
        ],
        on="alpha",
        suffixes=(
            "_fine",
            "_coarse",
        ),
        validate="one_to_one",
    )
)

for column in [
    "dynamic_profit",
    "myopic_profit",
    "vdo",
]:
    overlap_check[
        f"{column}_difference"
    ] = (
        overlap_check[
            f"{column}_fine"
        ]
        - overlap_check[
            f"{column}_coarse"
        ]
    )

difference_columns = [
    column
    for column in overlap_check.columns
    if column.endswith("_difference")
]

maximum_overlap_difference = float(
    overlap_check[
        difference_columns
    ].abs().max().max()
)

print(
    "Maximum fine-versus-coarse difference:",
    maximum_overlap_difference,
)

if maximum_overlap_difference > 1e-5:
    raise AssertionError(
        "The fine-grid run does not reproduce "
        "Notebook 10 at overlapping alpha values."
    )

display(
    overlap_check.round(8)
)

## 6. Identify local policy boundaries and their source

In [ ]:
def compare_schedule_maps(
    previous: Mapping[
        str,
        Sequence[float],
    ],
    current: Mapping[
        str,
        Sequence[float],
    ],
) -> dict[str, object]:
    all_products = sorted(
        set(previous)
        | set(current)
    )

    changed_products = []
    changed_cells = 0
    status_changes = 0
    depth_changes = 0

    for upc in all_products:
        previous_values = np.asarray(
            previous[upc],
            dtype=float,
        )
        current_values = np.asarray(
            current[upc],
            dtype=float,
        )

        changed = ~np.isclose(
            previous_values,
            current_values,
        )

        if changed.any():
            changed_products.append(
                str(upc)
            )
            changed_cells += int(
                changed.sum()
            )

        previous_positive = (
            previous_values > 0
        )
        current_positive = (
            current_values > 0
        )

        status_changes += int(
            np.sum(
                previous_positive
                != current_positive
            )
        )

        depth_changes += int(
            np.sum(
                previous_positive
                & current_positive
                & changed
            )
        )

    return {
        "changed_products": "|".join(
            changed_products
        ),
        "changed_product_count": len(
            changed_products
        ),
        "changed_cells": changed_cells,
        "status_changes": status_changes,
        "depth_changes": depth_changes,
    }


fine_results = fine_results.sort_values(
    "alpha"
).reset_index(drop=True)

fine_results[
    "dynamic_boundary"
] = (
    fine_results[
        "dynamic_schedule_signature"
    ]
    .ne(
        fine_results[
            "dynamic_schedule_signature"
        ].shift()
    )
)
fine_results[
    "myopic_boundary"
] = (
    fine_results[
        "myopic_schedule_signature"
    ]
    .ne(
        fine_results[
            "myopic_schedule_signature"
        ].shift()
    )
)

fine_results.loc[
    0,
    [
        "dynamic_boundary",
        "myopic_boundary",
    ],
] = False

transition_rows = []

for position in range(
    1,
    len(fine_results),
):
    previous_row = fine_results.iloc[
        position - 1
    ]
    current_row = fine_results.iloc[
        position
    ]

    if not (
        bool(
            current_row[
                "dynamic_boundary"
            ]
        )
        or bool(
            current_row[
                "myopic_boundary"
            ]
        )
    ):
        continue

    previous_key = (
        round(
            float(
                previous_row["alpha"]
            ),
            8,
        ),
        VALIDATION_CAPACITY,
    )
    current_key = (
        round(
            float(
                current_row["alpha"]
            ),
            8,
        ),
        VALIDATION_CAPACITY,
    )

    dynamic_change = compare_schedule_maps(
        fine_schedules[
            previous_key
        ][
            "dynamic"
        ],
        fine_schedules[
            current_key
        ][
            "dynamic"
        ],
    )

    myopic_change = compare_schedule_maps(
        fine_schedules[
            previous_key
        ][
            "myopic"
        ],
        fine_schedules[
            current_key
        ][
            "myopic"
        ],
    )

    dynamic_boundary = bool(
        current_row[
            "dynamic_boundary"
        ]
    )
    myopic_boundary = bool(
        current_row[
            "myopic_boundary"
        ]
    )

    if (
        dynamic_boundary
        and myopic_boundary
    ):
        boundary_type = "both"
    elif dynamic_boundary:
        boundary_type = "dynamic_only"
    else:
        boundary_type = "myopic_only"

    transition_rows.append(
        {
            "capacity": (
                VALIDATION_CAPACITY
            ),
            "alpha_previous": float(
                previous_row["alpha"]
            ),
            "alpha_current": float(
                current_row["alpha"]
            ),
            "boundary_type": (
                boundary_type
            ),
            "vdo_previous": float(
                previous_row["vdo"]
            ),
            "vdo_current": float(
                current_row["vdo"]
            ),
            "vdo_change": float(
                current_row["vdo"]
                - previous_row["vdo"]
            ),
            "vdo_percent_current": float(
                current_row[
                    "vdo_percent"
                ]
            ),
            "dynamic_promotion_count_previous": int(
                previous_row[
                    "dynamic_promotion_count"
                ]
            ),
            "dynamic_promotion_count_current": int(
                current_row[
                    "dynamic_promotion_count"
                ]
            ),
            "myopic_promotion_count_previous": int(
                previous_row[
                    "myopic_promotion_count"
                ]
            ),
            "myopic_promotion_count_current": int(
                current_row[
                    "myopic_promotion_count"
                ]
            ),
            "best_second_gap_current": float(
                current_row[
                    "best_second_gap"
                ]
            ),
            **{
                f"dynamic_{key}": value
                for key, value in (
                    dynamic_change.items()
                )
            },
            **{
                f"myopic_{key}": value
                for key, value in (
                    myopic_change.items()
                )
            },
        }
    )

boundary_transitions = (
    pd.DataFrame(
        transition_rows
    )
)

if not boundary_transitions.empty:
    boundary_transitions[
        "absolute_vdo_change"
    ] = (
        boundary_transitions[
            "vdo_change"
        ].abs()
    )

    boundary_transitions = (
        boundary_transitions
        .sort_values(
            "absolute_vdo_change",
            ascending=False,
        )
        .reset_index(drop=True)
    )

print(
    "Detected local boundaries:",
    len(boundary_transitions),
)

display(
    boundary_transitions
)

## 7. Boundary-validation figures

In [ ]:
fig, ax = plt.subplots(
    figsize=(9.4, 5.2)
)

ax.plot(
    fine_results["alpha"],
    fine_results["vdo_percent"],
    marker="o",
    markersize=4,
    linewidth=1.5,
    label="VDO",
)

dynamic_boundaries = (
    fine_results.loc[
        fine_results[
            "dynamic_boundary"
        ]
    ]
)
myopic_boundaries = (
    fine_results.loc[
        fine_results[
            "myopic_boundary"
        ]
    ]
)

ax.scatter(
    dynamic_boundaries["alpha"],
    dynamic_boundaries[
        "vdo_percent"
    ],
    marker="^",
    s=60,
    label="Dynamic schedule switch",
)

ax.scatter(
    myopic_boundaries["alpha"],
    myopic_boundaries[
        "vdo_percent"
    ],
    marker="s",
    s=44,
    label="Myopic schedule switch",
)

ax.axhline(
    0.0,
    linestyle=":",
    linewidth=1.0,
)

ax.set_xlabel(
    r"Contract generosity, $\alpha$"
)
ax.set_ylabel(
    "VDO relative to myopic profit (%)"
)
ax.set_title(
    "Local validation of the policy-boundary mechanism "
    f"(B={VALIDATION_CAPACITY})"
)
ax.legend()

fig.tight_layout()

BOUNDARY_FIGURE_PNG = (
    FIGURE_DIR
    / "boundary_validation_vdo.png"
)
BOUNDARY_FIGURE_PDF = (
    FIGURE_DIR
    / "boundary_validation_vdo.pdf"
)

fig.savefig(
    BOUNDARY_FIGURE_PNG,
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    BOUNDARY_FIGURE_PDF,
    bbox_inches="tight",
)

plt.show()

In [ ]:
fig, ax = plt.subplots(
    figsize=(9.4, 4.8)
)

ax.plot(
    fine_results["alpha"],
    fine_results[
        "best_second_gap"
    ],
    marker="o",
    markersize=4,
    linewidth=1.5,
)

for alpha in dynamic_boundaries[
    "alpha"
]:
    ax.axvline(
        alpha,
        linestyle="--",
        linewidth=0.8,
    )

ax.set_xlabel(
    r"Contract generosity, $\alpha$"
)
ax.set_ylabel(
    "Best minus second-best dynamic value"
)
ax.set_title(
    "Local dynamic-schedule separation"
)

fig.tight_layout()

GAP_FIGURE_PNG = (
    FIGURE_DIR
    / "boundary_validation_best_second_gap.png"
)
GAP_FIGURE_PDF = (
    FIGURE_DIR
    / "boundary_validation_best_second_gap.pdf"
)

fig.savefig(
    GAP_FIGURE_PNG,
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    GAP_FIGURE_PDF,
    bbox_inches="tight",
)

plt.show()

## 8. Save compact validation outputs

In [ ]:
RESULTS_PATH = (
    TABLE_DIR
    / "boundary_validation_results.csv"
)
TRANSITIONS_PATH = (
    TABLE_DIR
    / "boundary_validation_transitions.csv"
)
PRODUCT_DECOMPOSITION_PATH = (
    TABLE_DIR
    / "boundary_validation_product_decomposition.csv"
)
WEEKLY_DECOMPOSITION_PATH = (
    TABLE_DIR
    / "boundary_validation_weekly_decomposition.csv"
)
BOUNDARY_ARTIFACT_PATH = (
    PROCESSED_DIR
    / "boundary_validation_artifact.pkl"
)

fine_results.to_csv(
    RESULTS_PATH,
    index=False,
)
boundary_transitions.to_csv(
    TRANSITIONS_PATH,
    index=False,
)
fine_product_decomposition.to_csv(
    PRODUCT_DECOMPOSITION_PATH,
    index=False,
)
fine_weekly_decomposition.to_csv(
    WEEKLY_DECOMPOSITION_PATH,
    index=False,
)

boundary_artifact = {
    "capacity": (
        VALIDATION_CAPACITY
    ),
    "alpha_grid": (
        FINE_ALPHA_GRID
    ),
    "selected_washout": (
        SELECTED_WASHOUT
    ),
    "economic_profile_mode": (
        ECONOMIC_PROFILE_MODE
    ),
    "schedule_system": (
        fine_schedule_system
    ),
    "results": (
        fine_results
    ),
    "boundary_transitions": (
        boundary_transitions
    ),
    "schedules": (
        fine_schedules
    ),
    "product_decomposition": (
        fine_product_decomposition
    ),
    "weekly_decomposition": (
        fine_weekly_decomposition
    ),
    "overlap_check": (
        overlap_check
    ),
}

save_pickle(
    boundary_artifact,
    BOUNDARY_ARTIFACT_PATH,
)

print("Saved:", RESULTS_PATH)
print("Saved:", TRANSITIONS_PATH)
print("Saved:", PRODUCT_DECOMPOSITION_PATH)
print("Saved:", WEEKLY_DECOMPOSITION_PATH)
print("Saved:", BOUNDARY_ARTIFACT_PATH)

## Interpretation rule

The boundary mechanism is supported when:

- the fine-grid run exactly reproduces Notebook 10 at overlapping values;
- the dynamic policy remains weakly better than the myopic policy;
- a sharp local change in VDO coincides with a discrete change in at least one schedule;
- the transition table shows whether the jump is caused by the myopic policy switching first, the dynamic policy switching first, or both;
- the result is not driven by failed optimization or by the previously rejected realized weekly-cost profile.

After this notebook passes, the programming analysis is complete.